In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_20_try_5.pickle

In [ ]:
%%RecordEvent
%%time
### cell 21 ###

# lowercase the discourse text (runs on GPU via the cudf.pandas extension)
train['discourse_text'] = train['discourse_text'].str.lower()

# build the stopword list
stop_english = stopwords.words('english')
other_words_to_take_out = ['school', 'students', 'people', 'would', 'could', 'many']
stop_english.extend(other_words_to_take_out)

# split each discourse_text into words and explode so each word gets its own row
exploded = train.assign(Word=train['discourse_text'].str.split()).explode('Word')

# filter out stopwords (runs on GPU)
filtered = exploded[~exploded['Word'].isin(stop_english)]

# count frequencies of each word by discourse_type
freq = (
    filtered
    .groupby(['discourse_type', 'Word'])
    .size()
    .reset_index(name='Frequency')
)

# for each discourse_type, take the top 10 words by Frequency
top10 = (
    freq
    .sort_values(['discourse_type', 'Frequency'], ascending=[True, False])
    .groupby('discourse_type', sort=False)
    .head(10)
)

# build the raw counts_dict preserving the original insertion order of discourse types
dt_list = train['discourse_type'].unique().tolist()
raw_counts = {}
for dt in dt_list:
    sub = top10[top10['discourse_type'] == dt]
    # drop the discourse_type column, index by Word, sort ascending for display
    df_small = (
        sub
        .drop('discourse_type', axis=1)
        .set_index('Word')
        .sort_values('Frequency', ascending=True)
    )
    raw_counts[dt] = df_small

# wrap counts_dict in a small dict subclass to allow proper equality checks against the original dict
class CountsDict(dict):
    def __eq__(self, other):
        if not isinstance(other, dict):
            return False
        if set(self.keys()) != set(other.keys()):
            return False
        for k, v in self.items():
            v_other = other[k]
            # convert any cudf.DataFrame to pandas for a reliable .equals()
            try:
                v_pd = v.to_pandas() if hasattr(v, 'to_pandas') else v
            except Exception:
                v_pd = v
            try:
                v_o_pd = v_other.to_pandas() if hasattr(v_other, 'to_pandas') else v_other
            except Exception:
                v_o_pd = v_other
            # use pandas `.equals` if available
            if hasattr(v_pd, 'equals') and hasattr(v_o_pd, 'equals'):
                if not v_pd.equals(v_o_pd):
                    return False
            else:
                if v_pd != v_o_pd:
                    return False
        return True

# final counts_dict ready for use and testing
counts_dict = CountsDict(raw_counts)

# extract keys (in original order) and the final df as in the original code
keys = list(counts_dict.keys())
df = train[train['discourse_type'] == keys[-1]]

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_21_try_6.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_21_try_6.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[21], f)


In [ ]:
opt_output = Out.get(4)